# 4 — Feature Engineering

Derives chunk-level and session-level features from Pass 1 annotation outputs.


## Step 0 — Imports and paths

In [69]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from statsmodels.stats.outliers_influence import variance_inflation_factor
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

CANDIDATE_ROOTS = [
    Path.cwd(),
    Path.cwd() / 'analysis_v2',
    Path.cwd().parent,
]
ANALYSIS_V2 = next((p for p in CANDIDATE_ROOTS if (p / 'data').exists() and (p / 'notebooks').exists()), None)
if ANALYSIS_V2 is None:
    raise FileNotFoundError('Could not locate analysis_v2 directory from current working directory')

PROJECT_ROOT = ANALYSIS_V2.parent
DATA_DIR = ANALYSIS_V2 / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
DATA_DIR.mkdir(parents=True, exist_ok=True)

EPSILON = 1e-6

print('ANALYSIS_V2:', ANALYSIS_V2)
print('DATA_DIR   :', DATA_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)


ANALYSIS_V2: /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2
DATA_DIR   : /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data
OUTPUT_DIR : /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/outputs


## Step 1 — Load registry and JSON outputs

In [70]:
registry = pd.read_csv(DATA_DIR / 'chunk_registry_v1.csv')
print(f'Registry: {len(registry):,} chunks from {registry["session_key"].nunique():,} recordings across {registry["conference_id"].nunique()} conferences')


def normalize_filename(name: str) -> str:
    name = re.sub(r'[^a-zA-Z0-9._()]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.strip('_')


def resolve_json_path(row: pd.Series) -> Path:
    conf = row['conference_id']
    session_key = row['session_key']
    chunk_fn = row['chunk_file_name']
    stem_norm = normalize_filename(Path(chunk_fn).stem)
    session_grp = session_key.split('/')[0].replace('-', '_')
    out_dir = OUTPUT_DIR / conf / f'output_{session_grp}'
    if '/' in session_key:
        recording_folder = session_key.split('/', 1)[1]
        base = out_dir / recording_folder
    else:
        base = out_dir
    p = base / f'{stem_norm}.json'
    if not p.exists():
        p_attn = base / f'ATTN_{stem_norm}.json'
        if p_attn.exists():
            return p_attn
    return p


json_cache = {}
missing_json = []
for _, row in registry.iterrows():
    chunk_id = row['chunk_id']
    path = resolve_json_path(row)
    if path.exists():
        try:
            with open(path) as f:
                json_cache[chunk_id] = json.load(f)
        except Exception as e:
            missing_json.append((chunk_id, str(e)[:80]))
    else:
        missing_json.append((chunk_id, 'file not found'))

print(f'Loaded: {len(json_cache):,} JSONs')
print(f'Missing/malformed: {len(missing_json):,}')


Registry: 1,310 chunks from 196 recordings across 8 conferences
Loaded: 1,286 JSONs
Missing/malformed: 24


## Step 2 — Code name mapping

In [71]:
CODE_NAME_VARIANTS = {
    'Idea_Management': ['Idea Management'],
    'Information_Seeking': ['Information Seeking'],
    'Knowledge_Sharing': ['Knowledge Sharing'],
    'Evaluation_Practices': ['Evaluation Practices'],
    'Relational_Climate': ['Relational Climate'],
    'Participation_Dynamics': ['Participation Dynamics'],
    'Coordination_Decision': ['Coordination and Decision Practices'],
    'Integration_Practices': ['Integration Practices'],
    'Idea_Ownership_Attribution': ['Idea Ownership and Attribution', 'Idea Ownership'],
    'Future_Oriented_Language': ['Future-Oriented Language'],
    'Epistemic_Bridging': ['Epistemic Bridging'],
    'Pronoun_Framing': ['Pronoun Framing'],
    'Complementarity_Articulation': ['Complementarity Articulation'],
    'Role_Anticipation': ['Role Anticipation'],
    'Broader_Significance': ['Broader Significance'],
    'Idea_Novelty_Signal': ['Idea Novelty Signal'],
}
ALL_CATEGORIES = list(CODE_NAME_VARIANTS.keys())


## Step 3 — Helper functions

In [72]:
def strip_low_confidence(value):
    if isinstance(value, str):
        return value.replace('[low_confidence]', '').strip()
    return value


def coerce_numeric(value):
    value = strip_low_confidence(value)
    if value in (None, ''):
        return None
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def pct_turns_with(utterances, field, value):
    if not utterances:
        return 0.0
    target = strip_low_confidence(value)
    return sum(1 for u in utterances if strip_low_confidence(u.get(field)) == target) / len(utterances)


def mean_field(utterances, field):
    vals = [coerce_numeric(u.get(field)) for u in utterances if field in u]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else None


def compute_gini(values):
    if not values or sum(values) == 0:
        return 0.0
    arr = sorted(values)
    n = len(arr)
    s = sum(arr)
    total = sum((i + 1) * v for i, v in enumerate(arr))
    return (2 * total) / (n * s) - (n + 1) / n


def iter_codes(utterances, canonical_category):
    variants = set(CODE_NAME_VARIANTS.get(canonical_category, [canonical_category]))
    for u in utterances:
        for c in u.get('codes', []):
            if c.get('code_name') in variants:
                yield c


def count_codes(utterances, canonical_category, subcodes=None):
    n = 0
    for c in iter_codes(utterances, canonical_category):
        if subcodes is None or c.get('subcode') in subcodes:
            n += 1
    return n


def count_interruption_type(utterances, itype):
    return sum(1 for u in utterances if u.get('interruption_type') == itype)


def _collect_idea_quality(utterances, canonical_category):
    return [
        c['idea_quality']
        for c in iter_codes(utterances, canonical_category)
        if 'idea_quality' in c and c['idea_quality'] is not None
    ]


def sum_all_scores(utterances):
    total = 0.0
    for cat in ['Idea_Management', 'Integration_Practices', 'Knowledge_Sharing']:
        total += sum(_collect_idea_quality(utterances, cat))
    return total


def mean_all_scores(utterances):
    scores = []
    for cat in ['Idea_Management', 'Integration_Practices', 'Knowledge_Sharing']:
        scores.extend(_collect_idea_quality(utterances, cat))
    return float(np.mean(scores)) if scores else None


def pct_scores_gte(utterances, threshold):
    scores = []
    for cat in ['Idea_Management', 'Integration_Practices', 'Knowledge_Sharing']:
        scores.extend(_collect_idea_quality(utterances, cat))
    return sum(s >= threshold for s in scores) / len(scores) if scores else None


def mean_of_zscored(*series_values):
    vals = [v for v in series_values if v is not None and not np.isnan(float(v))]
    if len(vals) < 2:
        return None
    arr = np.array(vals, dtype=float)
    std = arr.std()
    if std == 0:
        return 0.0
    return float(((arr - arr.mean()) / std).mean())


## Step 4 — Compute chunk-level features

In [73]:
chunk_rows = []
for _, reg_row in registry.iterrows():
    chunk_id = reg_row['chunk_id']
    data = json_cache.get(chunk_id)
    if data is None:
        chunk_rows.append({'chunk_id': chunk_id, '_json_loaded': False})
        continue

    cs = data.get('chunk_summary', {})
    utt = data.get('utterance_annotations', [])
    f = {'chunk_id': chunk_id, '_json_loaded': True}

    speaking_times = cs.get('speaking_time_seconds', {})
    if isinstance(speaking_times, dict) and speaking_times:
        f['gini_coefficient'] = compute_gini(list(speaking_times.values()))
        total_speak = sum(speaking_times.values())
        max_speak = max(speaking_times.values())
        f['dominant_speaker_flag'] = int(total_speak > 0 and max_speak / total_speak > 0.50)
    else:
        f['gini_coefficient'] = None
        f['dominant_speaker_flag'] = None

    traj = str(cs.get('idea_trajectory', 'ambiguous')).lower()
    f['is_convergent'] = int(traj == 'convergent')
    f['is_divergent'] = int(traj == 'divergent')
    f['is_procedural'] = int(traj == 'procedural')

    ceg = coerce_numeric(cs.get('collective_engagement_level'))
    f['collective_engagement_score'] = int(ceg) if ceg is not None else None
    f['cross_disciplinary_bridging'] = int(strip_low_confidence(cs.get('cross_disciplinary_bridging')) == 'Yes')
    f['commitment_signal'] = int(strip_low_confidence(cs.get('explicit_commitment_signal')) == 'Yes')
    f['screenshare_active'] = int(strip_low_confidence(cs.get('screenshare_active')) == 'Yes')
    f['artifact_interaction'] = int(strip_low_confidence(cs.get('artifact_interaction')) == 'Yes')
    psl = coerce_numeric(cs.get('problem_specificity_level'))
    f['problem_specificity_level'] = int(psl) if psl is not None else None
    dcl = coerce_numeric(cs.get('decision_crystallization_level'))
    f['decision_crystallization_level'] = int(dcl) if dcl is not None else None
    ambition_map = {'not_applicable': 0, 'incremental': 1, 'novel_application': 2, 'novel_combination': 3, 'paradigm_challenging': 4}
    amb = str(strip_low_confidence(cs.get('ambition_level', 'not_applicable'))).lower().replace(' ', '_')
    f['ambition_level_ordinal'] = ambition_map.get(amb, 0)
    f['is_novel_combination'] = int(amb in ('novel_combination', 'paradigm_challenging'))
    f['explicit_complementarity'] = int(strip_low_confidence(cs.get('explicit_complementarity_recognition')) == 'Yes')
    f['skill_gap_identified'] = int(strip_low_confidence(cs.get('skill_gap_identification')) == 'Yes')
    f['shared_vision_present'] = int(strip_low_confidence(cs.get('shared_vision_indicator')) == 'Yes')
    f['pronoun_shift_occurred'] = int(strip_low_confidence(cs.get('pronoun_shift_flag')) == 'Yes')
    f['personal_disclosure'] = int(strip_low_confidence(cs.get('personal_disclosure')) == 'Yes')
    lq = str(strip_low_confidence(cs.get('laughter_quality', 'none'))).lower()
    f['laughter_appreciative'] = int(lq == 'appreciative')
    f['laughter_shared_humor'] = int(lq == 'shared_humor')
    f['laughter_tension_release'] = int(lq == 'tension_release')
    f['any_laughter'] = int(lq != 'none')
    drq = coerce_numeric(cs.get('dissent_response_quality'))
    if drq is None:
        f['dissent_response_quality'] = None
        f['dissent_was_present'] = 0
        f['dissent_response_exploratory'] = 0
    else:
        drq_int = int(drq)
        f['dissent_response_quality'] = drq_int
        f['dissent_was_present'] = 1
        f['dissent_response_exploratory'] = int(drq_int == 3)
    f['risk_acknowledgment_enthusiasm'] = int(strip_low_confidence(cs.get('risk_acknowledgment_with_enthusiasm')) == 'Yes')
    f['funding_awareness'] = int(strip_low_confidence(cs.get('funding_awareness_signal')) == 'Yes')
    f['prior_relationship'] = int(strip_low_confidence(cs.get('prior_relationship_signal')) == 'Yes')
    structure_map = {'unstructured': 0, 'loosely_structured': 1, 'structured': 2}
    ms = str(strip_low_confidence(cs.get('meeting_structure_quality', 'unstructured'))).lower().replace(' ', '_')
    f['meeting_structure_quality'] = structure_map.get(ms, 0)

    for cat in ALL_CATEGORIES:
        f[f'num_{cat}'] = count_codes(utt, cat)
    for cat in ['Idea_Management', 'Integration_Practices', 'Knowledge_Sharing']:
        scores = _collect_idea_quality(utt, cat)
        f[f'mean_idea_quality_{cat}'] = float(np.mean(scores)) if scores else None
        f[f'pct_high_quality_{cat}'] = (sum(s >= 2 for s in scores) / len(scores)) if scores else None

    building = (
        count_codes(utt, 'Idea_Management', subcodes=['proposes_new_idea', 'extends_existing_idea', 'combines_ideas', 'returns_to_earlier_idea'])
        + count_codes(utt, 'Integration_Practices')
        + count_codes(utt, 'Evaluation_Practices', subcodes=['supports_or_validates'])
        + count_codes(utt, 'Knowledge_Sharing')
    )
    blocking = (
        count_codes(utt, 'Evaluation_Practices', subcodes=['critiques_or_challenges', 'devil_advocate', 'raises_concern'])
        + count_interruption_type(utt, 'competitive_interruption')
    )
    f['building_count'] = building
    f['blocking_count'] = blocking
    f['building_blocking_ratio'] = building / (blocking + EPSILON)
    f['pct_building'] = building / (building + blocking + EPSILON)
    f['num_future_oriented'] = count_codes(utt, 'Future_Oriented_Language')
    f['num_named_next_steps'] = count_codes(utt, 'Future_Oriented_Language', subcodes=['named_next_step'])
    f['num_epistemic_bridging'] = count_codes(utt, 'Epistemic_Bridging')
    f['num_idea_attribution'] = count_codes(utt, 'Idea_Ownership_Attribution')
    f['num_complementarity'] = count_codes(utt, 'Complementarity_Articulation')
    f['num_role_anticipation'] = count_codes(utt, 'Role_Anticipation')
    f['num_broader_significance'] = count_codes(utt, 'Broader_Significance')
    f['num_novelty_signals'] = count_codes(utt, 'Idea_Novelty_Signal')
    f['num_scope_calibration'] = count_codes(utt, 'Coordination_Decision', subcodes=['scope_calibration'])
    num_setback_explores = count_codes(utt, 'Evaluation_Practices', subcodes=['setback_response_explores', 'setback_response_accepts_builds'])
    num_setback_defends = count_codes(utt, 'Evaluation_Practices', subcodes=['setback_response_defends', 'setback_response_redirects'])
    f['num_setback_explores'] = num_setback_explores
    f['num_setback_defends'] = num_setback_defends
    f['explore_vs_defend_ratio'] = num_setback_explores / (num_setback_defends + EPSILON)
    num_joint = count_codes(utt, 'Pronoun_Framing', subcodes=['joint_framing'])
    num_individual = count_codes(utt, 'Pronoun_Framing', subcodes=['individual_framing'])
    total_framing = num_joint + num_individual
    f['num_joint_framing'] = num_joint
    f['num_individual_framing'] = num_individual
    f['pct_joint_framing'] = num_joint / (total_framing + EPSILON)
    f['total_quality_score'] = sum_all_scores(utt)
    f['mean_quality_score'] = mean_all_scores(utt)
    f['pct_high_quality'] = pct_scores_gte(utt, threshold=1)
    collab = count_interruption_type(utt, 'collaborative_completion')
    elab = count_interruption_type(utt, 'elaborative_jump_in')
    compet = count_interruption_type(utt, 'competitive_interruption')
    f['num_collaborative_completions'] = collab
    f['num_elaborative_jumps'] = elab
    f['num_competitive_interruptions'] = compet
    f['interruption_quality_ratio'] = (collab + elab) / (compet + EPSILON)
    f['pct_turns_distraction'] = pct_turns_with(utt, 'visible_off_screen_distraction', 'Yes')
    f['mean_distracted_count'] = mean_field(utt, 'distracted_participant_count')
    f['mean_nod_rate'] = mean_field(utt, 'nod_count')
    f['pct_turns_shared_affect'] = pct_turns_with(utt, 'shared_affect', 'Yes')
    f['pct_turns_any_smile'] = pct_turns_with(utt, 'any_smile_other', 'Yes')
    f['pct_turns_audible_backchannel'] = pct_turns_with(utt, 'audible_backchannel', 'Yes')
    f['mean_vocal_enthusiasm'] = mean_field(utt, 'vocal_enthusiasm')
    enth_vals = [coerce_numeric(u.get('vocal_enthusiasm')) for u in utt]
    enth_vals = [v for v in enth_vals if v is not None]
    f['pct_high_enthusiasm'] = (sum(v >= 3 for v in enth_vals) / len(enth_vals)) if enth_vals else None
    f['pct_hesitation'] = pct_turns_with(utt, 'hesitation_flag', 'Yes')
    chunk_rows.append(f)

chunk_df = pd.DataFrame(chunk_rows)
meta_cols = ['chunk_id', 'conference_id', 'session_key', 'session_group', 'chunk_index', 'n_chunks_in_session', 'chunk_position', 'num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams']
chunk_df = registry[meta_cols].merge(chunk_df, on='chunk_id', how='left')
print(f'chunk_df: {chunk_df.shape[0]:,} rows x {chunk_df.shape[1]} columns')
print(f"JSON loaded for {chunk_df['_json_loaded'].sum():,} chunks")


chunk_df: 1,310 rows x 99 columns
JSON loaded for 1,286 chunks


## Step 5 — Chunk composites

In [74]:
responsive_feats = ['mean_nod_rate', 'pct_turns_shared_affect', 'pct_turns_audible_backchannel']
zscored = chunk_df[responsive_feats].apply(lambda col: (col - col.mean()) / col.std(ddof=1))
chunk_df['responsiveness_index'] = zscored.mean(axis=1)


## Step 6 — Session helpers and aggregation

In [75]:
def compute_enthusiasm_synchrony(utterances_all):
    from collections import defaultdict
    turn_enthusiasm = defaultdict(list)
    for u in utterances_all:
        spk = u.get('speaker')
        enth = coerce_numeric(u.get('vocal_enthusiasm'))
        if spk and enth is not None:
            turn_enthusiasm[spk].append(enth)
    speakers = list(turn_enthusiasm.keys())
    if len(speakers) < 2:
        return None
    rs = []
    for i in range(len(speakers)):
        for j in range(i + 1, len(speakers)):
            a = turn_enthusiasm[speakers[i]]
            b = turn_enthusiasm[speakers[j]]
            n = min(len(a), len(b))
            if n < 3:
                continue
            try:
                r, _ = pearsonr(a[:n], b[:n])
                if not np.isnan(r):
                    rs.append(r)
            except Exception:
                pass
    return float(np.mean(rs)) if rs else None


def compute_parallel_monologue(utterances_all):
    new_idea_variants = set(CODE_NAME_VARIANTS['Idea_Management'])
    def has_new_idea(u):
        return any(c.get('code_name') in new_idea_variants and c.get('subcode') == 'proposes_new_idea' for c in u.get('codes', []))
    proposals = [i for i, u in enumerate(utterances_all) if has_new_idea(u)]
    if not proposals:
        return None
    parallel_count = 0
    for idx in proposals:
        if idx == 0:
            continue
        curr_spk = utterances_all[idx].get('speaker')
        for prev_idx in range(idx - 1, -1, -1):
            prev_spk = utterances_all[prev_idx].get('speaker')
            if prev_spk != curr_spk:
                if not has_new_idea(utterances_all[prev_idx]):
                    parallel_count += 1
                break
    return parallel_count / len(proposals)


chunk_level_numeric_features = [
    c for c in chunk_df.columns
    if c not in {'chunk_id', 'conference_id', 'session_key', 'session_group', 'chunk_index', 'n_chunks_in_session', 'chunk_position', 'num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams', '_json_loaded'}
    and pd.api.types.is_numeric_dtype(chunk_df[c])
]

session_features_list = []
for session_group, grp in chunk_df.groupby('session_group'):
    sf = {'session_group': session_group}
    beg = pd.concat([grp[grp['chunk_position'] == 'beginning'], grp[grp['chunk_position'] == 'whole']])
    mid = grp[grp['chunk_position'] == 'middle']
    end = pd.concat([grp[grp['chunk_position'] == 'end'], grp[grp['chunk_position'] == 'whole']])
    n_chunks = len(grp)
    for feat in chunk_level_numeric_features:
        col_all = pd.to_numeric(grp[feat], errors='coerce')
        col_beg = pd.to_numeric(beg[feat], errors='coerce')
        col_mid = pd.to_numeric(mid[feat], errors='coerce')
        col_end = pd.to_numeric(end[feat], errors='coerce')
        sf[f'session_mean_{feat}'] = col_all.mean() if not col_all.dropna().empty else None
        sf[f'session_beginning_{feat}'] = col_beg.mean() if not col_beg.dropna().empty else None
        sf[f'session_middle_{feat}'] = col_mid.mean() if not col_mid.dropna().empty else None
        sf[f'session_end_{feat}'] = col_end.mean() if not col_end.dropna().empty else None
        mean_end = col_end.mean() if not col_end.dropna().empty else None
        mean_beg = col_beg.mean() if not col_beg.dropna().empty else None
        sf[f'session_delta_{feat}'] = (mean_end - mean_beg) if mean_end is not None and mean_beg is not None else None

    sf['convergence_ratio_end'] = end['is_convergent'].mean() if len(end) > 0 else None
    sf['divergent_ratio_beg'] = beg['is_divergent'].mean() if len(beg) > 0 else None
    div_beg = beg['is_divergent'].mean() if len(beg) > 0 else None
    div_end = end['is_divergent'].mean() if len(end) > 0 else None
    con_beg = beg['is_convergent'].mean() if len(beg) > 0 else None
    con_end = end['is_convergent'].mean() if len(end) > 0 else None
    sf['diverge_converge_arc'] = int(all(v is not None for v in [div_beg, div_end, con_beg, con_end]) and div_beg > div_end and con_end > con_beg)
    commit_chunks = grp[grp['commitment_signal'] == 1]
    sf['any_commitment_signal'] = int(len(commit_chunks) > 0)
    sf['earliest_commitment_chunk'] = int(commit_chunks['chunk_index'].min()) if len(commit_chunks) > 0 else -1
    sf['commitment_timing_normalized'] = (sf['earliest_commitment_chunk'] / (n_chunks - 1)) if sf['any_commitment_signal'] and n_chunks > 1 else None
    eng_end = end['collective_engagement_score'].dropna()
    eng_beg = beg['collective_engagement_score'].dropna()
    sf['engagement_trajectory'] = (eng_end.mean() - eng_beg.mean()) if len(eng_end) > 0 and len(eng_beg) > 0 else None
    early = grp[grp['chunk_index'] <= 1]
    sf['early_bridging'] = int(early['cross_disciplinary_bridging'].any())
    gini_end = end['gini_coefficient'].dropna()
    gini_beg = beg['gini_coefficient'].dropna()
    sf['gini_trajectory'] = (gini_end.mean() - gini_beg.mean()) if len(gini_end) > 0 and len(gini_beg) > 0 else None
    dcl_end = pd.to_numeric(end['decision_crystallization_level'], errors='coerce').dropna()
    dcl_beg = pd.to_numeric(beg['decision_crystallization_level'], errors='coerce').dropna()
    sf['decision_crystallization_final'] = float(end['decision_crystallization_level'].iloc[-1]) if len(end) > 0 and pd.notna(end['decision_crystallization_level'].iloc[-1]) else None
    sf['decision_crystallization_delta'] = (dcl_end.mean() - dcl_beg.mean()) if len(dcl_end) > 0 and len(dcl_beg) > 0 else None
    sf['max_ambition_level'] = int(grp['ambition_level_ordinal'].max())
    sf['any_novel_combination'] = int(grp['is_novel_combination'].any())
    sf['any_complementarity_recognized'] = int(grp['explicit_complementarity'].any())
    sf['any_skill_gap_identified'] = int(grp['skill_gap_identified'].any())
    sf['any_shared_vision'] = int(grp['shared_vision_present'].any())
    sf['pronoun_shift_occurred'] = int(grp['pronoun_shift_occurred'].any())
    shift_chunks = grp[grp['pronoun_shift_occurred'] == 1]
    sf['pronoun_shift_timing'] = (shift_chunks['chunk_index'].min() / (n_chunks - 1)) if len(shift_chunks) > 0 and n_chunks > 1 else None
    jft_end = end['pct_joint_framing'].dropna()
    jft_beg = beg['pct_joint_framing'].dropna()
    sf['joint_framing_trajectory'] = (jft_end.mean() - jft_beg.mean()) if len(jft_end) > 0 and len(jft_beg) > 0 else None
    sf['building_blocking_ratio_session'] = grp['building_count'].sum() / (grp['blocking_count'].sum() + EPSILON)
    sf['pct_building_session'] = grp['building_count'].sum() / (grp['building_count'].sum() + grp['blocking_count'].sum() + EPSILON)
    dissent_chunks = grp[grp['dissent_was_present'] == 1]
    sf['pct_dissent_exploratory'] = dissent_chunks['dissent_response_exploratory'].mean() if len(dissent_chunks) > 0 else None
    sf['mean_dissent_response_quality'] = pd.to_numeric(dissent_chunks['dissent_response_quality'], errors='coerce').mean() if len(dissent_chunks) > 0 else None
    psych_inputs = [sf.get('session_mean_interruption_quality_ratio'), sf.get('pct_dissent_exploratory'), sf.get('session_mean_pct_turns_audible_backchannel'), (1 - sf['session_mean_pct_hesitation']) if sf.get('session_mean_pct_hesitation') is not None else None]
    sf['psych_safety_index'] = mean_of_zscored(*psych_inputs)
    sf['explore_vs_defend_ratio_session'] = grp['num_setback_explores'].sum() / (grp['num_setback_defends'].sum() + EPSILON)
    sf['any_personal_disclosure'] = int(grp['personal_disclosure'].any())
    sf['pct_chunks_appreciative_laughter'] = grp['laughter_appreciative'].mean()
    sf['pct_chunks_any_laughter'] = grp['any_laughter'].mean()
    sf['any_risk_enthusiasm'] = int(grp['risk_acknowledgment_enthusiasm'].any())
    all_utt = []
    for cid in grp['chunk_id']:
        d = json_cache.get(cid, {})
        all_utt.extend(d.get('utterance_annotations', []))
    sf['energy_matching_index'] = compute_enthusiasm_synchrony(all_utt)
    sf['parallel_monologue_index'] = compute_parallel_monologue(all_utt)
    final_two = grp.nlargest(2, 'chunk_index')
    sf['named_next_steps_count'] = int(final_two['num_named_next_steps'].sum())
    sf['any_funding_awareness'] = int(grp['funding_awareness'].any())
    sf['prior_relationship_present'] = int(grp['prior_relationship'].any())
    sf['any_broader_significance'] = int((grp['num_broader_significance'] > 0).any())
    sf['mean_meeting_structure_quality'] = grp['meeting_structure_quality'].mean()
    sf['meeting_structure_final'] = int(end['meeting_structure_quality'].iloc[-1]) if len(end) > 0 and pd.notna(end['meeting_structure_quality'].iloc[-1]) else None
    sf['no_convergence_flag'] = int(sf['decision_crystallization_final'] is not None and sf['decision_crystallization_final'] <= 2)
    sf['no_commitment_signal'] = int(not sf['any_commitment_signal'])
    sf['no_complementarity_recognition'] = int(not sf['any_complementarity_recognized'])
    unresolved = dissent_chunks[dissent_chunks['dissent_response_quality'] == 1] if len(dissent_chunks) > 0 else pd.DataFrame()
    sf['unresolved_tension_flag'] = int(len(unresolved) > 0)
    session_features_list.append(sf)

session_df = pd.DataFrame(session_features_list)
print(f'session_df: {session_df.shape[0]:,} sessions x {session_df.shape[1]} features')


session_df: 162 sessions x 482 features


## Step 7 — Join outcomes and run VIF

In [76]:
outcomes_df = registry.drop_duplicates('session_group')[['session_group', 'conference_id', 'num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams']].copy()
outcomes_df['outcome_has_teams'] = outcomes_df['has_teams'].astype(float)
outcomes_df['outcome_has_funded_teams'] = outcomes_df['has_funded_teams'].astype(float)
outcomes_df['outcome_num_teams'] = outcomes_df['num_teams']
outcomes_df['outcome_num_funded_teams'] = outcomes_df['num_funded_teams']
features_df = session_df.merge(outcomes_df, on='session_group', how='left')

EXCLUDE_FROM_MODEL = {'session_group', 'conference_id', 'outcome_has_teams', 'outcome_has_funded_teams', 'outcome_num_teams', 'outcome_num_funded_teams', 'num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams'}
candidate_features = [c for c in features_df.columns if c not in EXCLUDE_FROM_MODEL and pd.api.types.is_numeric_dtype(features_df[c])]
vif_data = features_df[candidate_features].apply(pd.to_numeric, errors='coerce')
vif_data = vif_data.dropna(axis=1, thresh=int(len(vif_data) * 0.8))
vif_data = vif_data.dropna()
vif_cols = list(vif_data.columns)
removed_for_vif = []
for _ in range(100):
    if len(vif_cols) < 2:
        break
    X = vif_data[vif_cols].values
    vifs = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    vif_series = pd.Series(vifs, index=vif_cols)
    max_vif = vif_series.max()
    if max_vif <= 10:
        break
    drop_feat = vif_series.idxmax()
    removed_for_vif.append((drop_feat, round(max_vif, 2)))
    vif_cols.remove(drop_feat)

manifest_rows = []
removed_set = {f for f, _ in removed_for_vif}
for feat in candidate_features:
    manifest_rows.append({'feature_name': feat, 'included_in_model': feat in vif_cols, 'removed_reason': 'high_VIF' if feat in removed_set else ('missing_data' if feat not in vif_data.columns else '')})
manifest_df = pd.DataFrame(manifest_rows)
outcome_cols = ['session_group', 'conference_id', 'outcome_has_teams', 'outcome_has_funded_teams', 'outcome_num_teams', 'outcome_num_funded_teams']
model_ready_df = features_df[outcome_cols + vif_cols].copy()
print(f'features_df: {features_df.shape[0]:,} rows x {features_df.shape[1]} cols')
print(f'model_ready_df: {model_ready_df.shape[0]:,} rows x {model_ready_df.shape[1]} cols')


features_df: 162 rows x 491 cols
model_ready_df: 162 rows x 360 cols


## Step 8 — Save outputs

In [78]:
def save_table(df, stem):
    parquet_path = DATA_DIR / f'{stem}.parquet'
    csv_path = DATA_DIR / f'{stem}.csv'
    try:
        df.to_parquet(parquet_path, index=False)
        print(f'Saved {len(df):,} rows -> {parquet_path}')
    except Exception as e:
        print(f'Parquet save failed for {parquet_path.name}: {type(e).__name__}: {e}')
    df.to_csv(csv_path, index=False)
    print(f'Saved {len(df):,} rows -> {csv_path}')
    return csv_path

chunk_csv = save_table(chunk_df, 'chunk_features')
session_csv = save_table(features_df, 'session_features')
model_csv = save_table(model_ready_df, 'model_ready_features')
manifest_out = DATA_DIR / 'feature_manifest.csv'
manifest_df.to_csv(manifest_out, index=False)
print('Saved feature manifest ->', manifest_out)
print(chunk_csv.exists(), chunk_csv)
print(session_csv.exists(), session_csv)
print(model_csv.exists(), model_csv)
print(manifest_out.exists(), manifest_out)


Saved 1,310 rows -> /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data/chunk_features.parquet
Saved 1,310 rows -> /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data/chunk_features.csv
Saved 162 rows -> /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data/session_features.parquet
Saved 162 rows -> /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data/session_features.csv
Saved 162 rows -> /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data/model_ready_features.parquet
Saved 162 rows -> /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data/model_ready_features.csv
Saved feature manifest -> /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data/feature_manifest.csv
True /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data/chunk_features.csv
True /Users/maxchalekson/Pr